In [2]:
import numpy as np

# Load data form scratch

from transcription_pipeline import preprocessing_pipeline
from tqdm import tqdm

pipeline_path ='/mnt/Data6/Enze/transcription_pipeline'

# IMPORTANT: File path has to be relative to Jupyter Notebook.
folder_list = [
    "/Data/2025-07-15-Enze/hbP2delta4_MCP-mSG_His-RFP003",
    "/Data/2025-07-15-Enze/hbP2delta4_MCP-mSG_His-RFP004",
    "/Data/2025-07-15-Enze/hbP2delta4_MCP-mSG_His-RFP005",
    "/Data/2025-07-15-Enze/hbP2delta4_MCP-mSG_His-RFP006",
    "/Data/2025-07-15-Enze/hbP2delta4_MCP-mSG_His-RFP007",
    "/Data/2025-07-15-Enze/hbP2delta4_MCP-mSG_His-RFP008",
    "/Data/2025-07-15-Enze/hbP2delta4_MCP-mSG_His-RFP009",
    "/Data/2025-07-15-Enze/hbP2delta4_MCP-mSG_His-RFP010",
    "/Data/2025-07-15-Enze/hbP2delta4_MCP-mSG_His-RFP011"
    ]
dataset_list = [pipeline_path + folder for folder in folder_list]

for dataset_name in tqdm(dataset_list):
    # Imports dataset into memory, trimming the last frame of each time-series as requested
    dataset = preprocessing_pipeline.DataImport(
        name_folder=dataset_name,
        # trim_series=True,
        # registration_channel=1,
        working_storage_mode="zarr",
        zarr_chunk_memory_size="1 GB",
        # import_previous=True,
    )

    # Saves each channel of the dataset as a zarr, and pickles together dictionaries
    # of relevant metadata
    dataset.save()

100%|██████████| 9/9 [19:32<00:00, 130.25s/it]


In [ ]:
# Load data from zarr

from transcription_pipeline import preprocessing_pipeline
from tqdm import tqdm

pipeline_path ='/mnt/Data6/Enze/transcription_pipeline'

folder_path = "/Data/2025-06-17-Enze/hbP2WT_MCP-mSG_His-RFP006"

dataset_path = pipeline_path + folder_path

dataset = preprocessing_pipeline.DataImport(
    name_folder=dataset_path,
    # trim_series=True,
    # registration_channel=1,
    # working_storage_mode="zarr",
    # zarr_chunk_memory_size="1 GB",
    import_previous=True,
)

In [ ]:
# Start Dask client for parallelization
from dask.distributed import LocalCluster, Client

cluster = LocalCluster(
    host="localhost",
    # scheduler_port=8786,
    threads_per_worker=1,
    n_workers=14,
    memory_limit="20GB",
)

client = Client(cluster)

In [ ]:
client

In [ ]:
client.dashboard_link

In [ ]:
# Run spot segmentation, filtering and quantification and transfer nuclear labels
# over to corresponding spots.
from transcription_pipeline import spot_pipeline

In [ ]:
# Need to add a memory sink for label expansion, otherwise nuclear labels
# get pulled into memory whole

In [ ]:
# Spot tracking setting from Nick

spot_channel = 0

spot_tracking = spot_pipeline.Spot(
    data=dataset.channels_full_dataset[spot_channel],
    global_metadata=dataset.export_global_metadata[spot_channel],
    frame_metadata=dataset.export_frame_metadata[spot_channel],
    expand_distance=3,
    search_range_um=2.5,    # how far you expect the spots move between frames
    retrack_search_range_um=4.5,
    spot_sigma_z_bounds=(0.16, 100),
    spot_sigma_x_y_bounds=(0.052, 100),
    threshold_factor=2.5,   # balance between background and real signal (triangle threshold) may play with this value
    memory=3,
    retrack_after_filter=True,
    min_track_length=0,
    series_splits=dataset.series_splits,
    series_shifts=dataset.series_shifts,
    keep_bandpass=False,
    keep_futures=False,
    keep_spot_labels=False,
    evaluate=True,
    retrack_by_intensity=False,
    client=client,
)

spot_tracking.extract_spot_traces(
    working_memory_mode="zarr",
    working_memory_folder=dataset_path,
)


# Saves tracked spot mask as a zarr, and pickles dataframes with spot fitting and
# quantification information.
spot_tracking.save_results(
    name_folder=dataset_path, save_array_as=None
)

In [ ]:
# Load existing spot segmentation with different parameter set

%%time

spot_tracking = spot_pipeline.Spot(
    data=dataset.channels_full_dataset[0],
    global_metadata=dataset.export_global_metadata[0],
    frame_metadata=dataset.export_frame_metadata[0],
    labels=nuclear_tracking.reordered_labels,
    max_num_spots=2,
    expand_distance=3,
    search_range_um=1.5,
    retrack_search_range_um=3,
    memory=3,
    min_track_length=1,
    series_splits=dataset.series_splits,
    series_shifts=dataset.series_shifts,
    keep_bandpass=False,
    keep_futures=False,
    keep_spot_labels=False,
    evaluate=False,
    client=client,
)

spot_tracking.read_results(
    name_folder="embryo2",
    import_all=True,
)

In [ ]:
# Load tracking data

spot_tracking = spot_pipeline.Spot()
spot_tracking.read_results(name_folder=dataset_path)

spot_df = spot_tracking.spot_dataframe

In [ ]:
# Plot single trace

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def plot_single_trace(label_id):
    spot_trace = spot_df[spot_df["particle"] == label_id].sort_values("t_s")
    time = spot_trace["t_s"]/60
    signal = spot_trace["intensity_from_neighborhood"]

    plt.plot(time, signal, 'o')
    plt.title(f"Signal v.s. Time(min), Particle {label_id}")
    plt.xlabel("Time(min)")
    plt.ylabel("Signal")
    plt.show()
    plt.close()

In [ ]:
plot_single_trace(374)

In [ ]:
# Get all the traces and sort by length

In [ ]:
import pandas as pd

In [ ]:
def get_single_trace(label_id):
    spot_trace = spot_df[spot_df["particle"] == label_id].sort_values("t_s")
    time = spot_trace["t_s"]
    signal = spot_trace["intensity_from_neighborhood"]
    return time, signal

In [ ]:
num_particles = spot_df["particle"].max()
traces = pd.DataFrame({})

for label_id in range(num_particles):
    trace_time, trace_signal = get_single_trace(label_id)
    trace = pd.DataFrame({"particle" : label_id, "t_s" : [trace_time.to_numpy()], "intensity_from_neighborhood" : [trace_signal.to_numpy()], "length" : len(trace_time)})
    traces = pd.concat([traces,trace], ignore_index=True)

traces_sorted = traces.sort_values("length", ascending=False)

In [ ]:
traces_sorted

In [ ]:
example_particle = 61

In [ ]:
traces_sorted[traces_sorted["particle"] == example_particle]

In [ ]:
plot_single_trace(example_particle)